# CHARMM36 lipid-bilayer analysis

All-atom CHARMM36 binary bilayers: **DOPC/DOPA, DOPC/DOPG, DOPC/DLPG, DOPC/DLPA**.

Eight properties, each in its own section. The trajectory is loaded **once** and
reused. Run the setup + config + load cells first, then any analysis section.

**CHARMM36 conventions used:** sn-1 chain = `C3x`, sn-2 = `C2x` (carbonyls
`C21`/`C31` excluded), headgroup marker = phosphate `P`, TIP3 water oxygen = `OH2`.


## 0 &nbsp; Setup

In [ ]:
# Run once. In Jupyter, '!' runs a shell command.
# !pip install "MDAnalysis>=2.4" numpy scipy matplotlib
%matplotlib inline
import os, re, itertools
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

## 1 &nbsp; Config

Edit these to match your run. For the **structural** analyses use a whole/centred
trajectory; for **diffusion** use a no-jump trajectory (see the preprocessing note
at the bottom). Set `LIPIDS` to the two residue names of the system you're on.


In [ ]:
TOPOLOGY   = "md.tpr"          # .tpr preferred (has bonds+masses); .gro/.pdb ok for geometry
TRAJ_STRUCT= "md_center.xtc"   # whole & centred trajectory (structural analyses)
TRAJ_NOJUMP= "md_nojump.xtc"   # pbc nojump trajectory (diffusion only)

# The two lipids in THIS system, e.g. ["DOPC","DOPA"]. Others are harmless if absent.
LIPIDS     = ["DOPC", "DOPA", "DOPG", "DLPG", "DLPA"]

HEAD       = "P"                          # headgroup / leaflet marker
WATER_SEL  = "resname TIP3 SOL"           # water residues
WATER_O    = "name OH2 or name OW or name O"   # water oxygen
STEP       = 1                            # frame stride (raise to subsample)
OUTDIR     = "results"
os.makedirs(OUTDIR, exist_ok=True)

## 2 &nbsp; Load the trajectory (once)

In [ ]:
u = mda.Universe(TOPOLOGY, TRAJ_STRUCT)
has_bonds = len(getattr(u, "bonds", [])) > 0
present = [r for r in LIPIDS if len(u.select_atoms(f"resname {r}")) > 0]
print(f"{len(u.atoms)} atoms, {len(u.trajectory)} frames. bonds={has_bonds}")
print("Lipids present:", present)

## 3 &nbsp; Shared helpers

In [ ]:
def carbons_for_chain(residue, prefix):
    """Acyl carbons of one chain ('C3'=sn-1, 'C2'=sn-2), skipping carbonyl (idx 1)."""
    pat = re.compile(r"^" + prefix + r"(\d+)$")
    hits = [(int(m.group(1)), at) for at in residue.atoms
            if (m := pat.match(at.name)) and int(m.group(1)) >= 2]
    return sorted(hits, key=lambda t: t[0])

def bonded_hydrogens(carbon):
    if has_bonds:
        return [a for a in carbon.bonded_atoms if a.name.startswith("H")]
    return list(u.select_atoms("name H* and around 1.3 group c",
                               c=u.atoms[[carbon.index]]))

## 4 &nbsp; Order parameter (−S_CD)

Per-carbon deuterium order parameter for each chain; C–H bonds auto-detected.


In [ ]:
order = {}   # (res, tag) -> (indices, -Scd, sem)
for res in present:
    for prefix, tag in (("C3","sn1"), ("C2","sn2")):
        first = u.select_atoms(f"resname {res}").residues[0]
        carbons = carbons_for_chain(first, prefix)
        if not carbons:
            continue
        cidx = [i for i,_ in carbons]
        acc = {i: [] for i in cidx}
        for ts in u.trajectory[::STEP]:
            for resid in u.select_atoms(f"resname {res}").residues:
                for idx, catom in carbons_for_chain(resid, prefix):
                    hs = bonded_hydrogens(catom)
                    if not hs: continue
                    cpos = catom.position
                    vals = []
                    for h in hs:
                        v = h.position - cpos; v /= np.linalg.norm(v)
                        vals.append(0.5*(3*v[2]*v[2] - 1))
                    acc[idx].append(np.mean(vals))
        idxs, scd, sem = [], [], []
        for i in cidx:
            a = np.array(acc[i])
            if a.size:
                idxs.append(i); scd.append(-a.mean()); sem.append(a.std()/np.sqrt(a.size))
        order[(res,tag)] = (np.array(idxs), np.array(scd), np.array(sem))
        np.savetxt(f"{OUTDIR}/order_{res}_{tag}.dat",
                   np.column_stack([idxs,scd,sem]), header="C_index -SCD sem")

plt.figure(figsize=(7,5))
for (res,tag),(i,s,e) in order.items():
    plt.errorbar(i, s, yerr=e, marker="o", capsize=2, label=f"{res} {tag}")
plt.xlabel("Carbon number"); plt.ylabel(r"$-S_{CD}$")
plt.title("Order parameters"); plt.legend(fontsize=8); plt.grid(alpha=.3); plt.show()

## 5 &nbsp; Interaction energies (lipid–lipid + partial Coul/LJ)

GROMACS gets pair energies from a **rerun with energy groups** — that part runs in
a terminal, not the notebook (energy-group decomposition needs `-nb cpu`):

```bash
# 1) add to a copy of your mdp:   energygrps = DOPC DOPA
gmx grompp -f rerun.mdp -c md.gro -p topol.top -n index.ndx -o rerun.tpr -maxwarn 2
gmx mdrun  -s rerun.tpr -rerun md_center.xtc -e ener.edr -nb cpu
# 2) extract the cross terms:
gmx energy -f ener.edr -o inter_DOPC-DOPA.xvg   # pick Coul-SR:DOPC-DOPA  LJ-SR:DOPC-DOPA
```

The cell below reads the resulting `.xvg` and reports mean ± SEM of the electrostatic
(Coul-SR), van der Waals (LJ-SR) and total lipid–lipid energy.


In [ ]:
def read_xvg(path):
    legends, rows = [], []
    for line in open(path):
        if line.startswith("@") and " legend " in line:
            legends.append(line.split('"')[1])
        elif not line.startswith(("#","@")):
            rows.append([float(x) for x in line.split()])
    return legends, np.array(rows)

def summarise_energy(xvg):
    leg, d = read_xvg(xvg)
    print(f"{'term':<26}{'mean':>12}{'SEM':>12}  (kJ/mol)")
    tot = np.zeros(d.shape[0])
    for i,name in enumerate(leg):
        c = d[:,i+1]; print(f"{name:<26}{c.mean():12.3f}{c.std()/np.sqrt(len(c)):12.3f}")
        if "SR" in name: tot += c
    if len(leg) > 1:
        print(f"{'SUM (lipid-lipid)':<26}{tot.mean():12.3f}{tot.std()/np.sqrt(len(tot)):12.3f}")

# summarise_energy(f"{OUTDIR}/inter_DOPC-DOPA.xvg")   # uncomment once you have the xvg

## 6 &nbsp; Lateral spatial distribution (per leaflet)

In [ ]:
BINS = 60
maps, counts = {}, {}
for r in present:
    for leaf in ("upper","lower"):
        maps[(r,leaf)] = np.zeros((BINS,BINS)); counts[(r,leaf)] = 0
allP = u.select_atoms(f"name {HEAD} and resname " + " ".join(present))
for ts in u.trajectory[::STEP]:
    Lx, Ly = ts.dimensions[:2]
    zmid = allP.positions[:,2].mean()
    ex = np.linspace(0,Lx,BINS+1); ey = np.linspace(0,Ly,BINS+1)
    for r in present:
        p = u.select_atoms(f"name {HEAD} and resname {r}").positions
        for leaf, m in (("upper", p[:,2]>=zmid), ("lower", p[:,2]<zmid)):
            q = p[m]
            if len(q)==0: continue
            H,_,_ = np.histogram2d(np.mod(q[:,0],Lx), np.mod(q[:,1],Ly), bins=[ex,ey])
            maps[(r,leaf)] += H; counts[(r,leaf)] += 1
keys = [k for k in maps if counts[k]>0]
cols = min(4,len(keys)) or 1; rows = (len(keys)+cols-1)//cols
fig, ax = plt.subplots(rows, cols, figsize=(4*cols,3.4*rows), squeeze=False)
for a,(r,leaf) in zip(ax.flat, keys):
    im = a.imshow((maps[(r,leaf)]/counts[(r,leaf)]).T, origin="lower", cmap="viridis")
    a.set_title(f"{r} {leaf}"); fig.colorbar(im, ax=a, shrink=.8)
for a in ax.flat[len(keys):]: a.axis("off")
plt.tight_layout(); plt.show()

## 7 &nbsp; Voronoi analysis (area per lipid, neighbours)

Periodic 2D tessellation of phosphates per leaflet → APL, mean neighbour count,
and like-neighbour fraction (a clustering/mixing signal).


In [ ]:
from scipy.spatial import Voronoi

def poly_area(pts):
    x,y = pts[:,0], pts[:,1]
    return 0.5*abs(np.dot(x,np.roll(y,1)) - np.dot(y,np.roll(x,1)))

def tile(points, Lx, Ly):
    shifts = [(i,j) for i in (-1,0,1) for j in (-1,0,1)]
    tp = [points + np.array([sx*Lx, sy*Ly]) for sx,sy in shifts]
    idx = [np.arange(len(points)) for _ in shifts]
    return np.vstack(tp), np.concatenate(idx)

def leaflet_voronoi(pts, rtypes, Lx, Ly):
    N = len(pts); tp, tmap = tile(pts, Lx, Ly); vor = Voronoi(tp)
    central = np.arange(4*N, 5*N)
    areas = np.full(N, np.nan)
    for k, pidx in enumerate(central):
        reg = vor.regions[vor.point_region[pidx]]
        if reg and -1 not in reg:
            areas[k] = poly_area(vor.vertices[reg])
    neigh = [set() for _ in range(N)]
    for i,j in vor.ridge_points:
        if 4*N<=i<5*N: neigh[i-4*N].add(tmap[j])
        if 4*N<=j<5*N: neigh[j-4*N].add(tmap[i])
    for k in range(N): neigh[k].discard(k)
    nn = np.array([len(neigh[k]) for k in range(N)], float)
    like = np.array([np.mean([rtypes[m]==rtypes[k] for m in neigh[k]]) if neigh[k]
                     else np.nan for k in range(N)])
    return areas, nn, like

P = u.select_atoms(f"name {HEAD} and resname " + " ".join(present))
rt_all = np.array([a.resname for a in P])
acc = {r: {"apl":[], "nn":[], "like":[]} for r in present}
for ts in u.trajectory[::STEP]:
    Lx, Ly = ts.dimensions[:2]
    pos = P.positions.copy(); pos[:,0]%=Lx; pos[:,1]%=Ly
    zmid = pos[:,2].mean()
    for mask in (pos[:,2]>=zmid, pos[:,2]<zmid):
        sub = pos[mask][:,:2]; rsub = rt_all[mask]
        if len(sub) < 4: continue
        ar, nn, lk = leaflet_voronoi(sub, rsub, Lx, Ly)
        for r in present:
            m = rsub==r
            acc[r]["apl"] += list(ar[m][~np.isnan(ar[m])])
            acc[r]["nn"]  += list(nn[m])
            acc[r]["like"]+= list(lk[m][~np.isnan(lk[m])])
print(f"{'lipid':<8}{'APL(A^2)':>10}{'SEM':>8}{'<neigh>':>10}{'like':>8}")
for r in present:
    apl=np.array(acc[r]["apl"]); nn=np.array(acc[r]["nn"]); lk=np.array(acc[r]["like"])
    if apl.size:
        print(f"{r:<8}{apl.mean():10.2f}{apl.std()/np.sqrt(apl.size):8.2f}{nn.mean():10.2f}{lk.mean():8.3f}")
# sanity check vs box estimate:
print("box-estimate APL = 2*Lx*Ly/Nlipids =",
      round(2*Lx*Ly/len(P.residues),2), "A^2 (mixture-averaged)")

## 8 &nbsp; Water depth / penetration

In [ ]:
CORE = 8.0   # half-thickness (A) of hydrophobic core for the water count
Pat = u.select_atoms(f"name {HEAD}")
wat = u.select_atoms(f"({WATER_SEL}) and ({WATER_O})")
if len(wat)==0: wat = u.select_atoms(WATER_SEL)
zmax, NB = 60.0, 100
edges = np.linspace(-zmax, zmax, NB+1); ctr = 0.5*(edges[1:]+edges[:-1])
pw = np.zeros(NB); pp = np.zeros(NB); nf = 0; depths = []; core = []
for ts in u.trajectory[::STEP]:
    zc = Pat.positions[:,2].mean(); zP = Pat.positions[:,2]
    upP = zP[zP>=zc].mean(); loP = zP[zP<zc].mean()
    zw = wat.positions[:,2]
    pw += np.histogram(zw-zc, bins=edges)[0]
    pp += np.histogram(zP-zc, bins=edges)[0]
    pen = np.concatenate([upP - zw[zw>=zc], zw[zw<zc] - loP])
    depths.append(pen.max() if pen.size else np.nan)
    core.append(int(np.sum(np.abs(zw-zc) < CORE))); nf += 1
pw/=nf; pp/=nf; depths=np.array(depths); core=np.array(core,float)
print(f"deepest water past head plane: {np.nanmean(depths):.2f} +/- {np.nanstd(depths):.2f} A")
print(f"#waters within +/-{CORE} A of centre: {core.mean():.2f} +/- {core.std():.2f}")
plt.figure(figsize=(7,4))
plt.plot(ctr, pw, label="water"); plt.plot(ctr, pp, label="phosphate")
plt.xlabel("z rel. bilayer centre (A)"); plt.ylabel("mean count/bin")
plt.legend(); plt.grid(alpha=.3); plt.title("Water penetration"); plt.show()

## 9 &nbsp; Lipid tail splay

In [ ]:
def chain_ends(res, prefix):
    cs = carbons_for_chain(res, prefix)
    return (cs[0][1], cs[-1][1]) if len(cs)>=2 else None
cache = {}
for r in present:
    cache[r] = []
    for res in u.select_atoms(f"resname {r}").residues:
        e2, e3 = chain_ends(res,"C2"), chain_ends(res,"C3")
        if e2 and e3:
            cache[r].append((e2[0].index,e2[1].index,e3[0].index,e3[1].index))
angles = {r: [] for r in present}
for ts in u.trajectory[::STEP]:
    pos = u.atoms.positions
    for r in present:
        for i2,j2,i3,j3 in cache[r]:
            v2 = pos[j2]-pos[i2]; v1 = pos[j3]-pos[i3]
            n2, n1 = np.linalg.norm(v2), np.linalg.norm(v1)
            if n1 and n2:
                angles[r].append(np.degrees(np.arccos(np.clip(np.dot(v1,v2)/(n1*n2),-1,1))))
plt.figure(figsize=(7,4))
for r in present:
    a = np.array(angles[r])
    if a.size:
        print(f"{r}: <splay> = {a.mean():.1f} +/- {a.std():.1f} deg")
        plt.hist(a, bins=60, density=True, histtype="step", label=r)
plt.xlabel("intra-lipid tail splay angle (deg)"); plt.ylabel("P(theta)")
plt.legend(); plt.grid(alpha=.3); plt.title("Tail splay"); plt.show()

## 10 &nbsp; Radial distribution function (2D lateral, P–P)

In [ ]:
RMAX_nm, NBINS = 3.0, 150
rmax = RMAX_nm*10; edges = np.linspace(0,rmax,NBINS+1)
ctr = 0.5*(edges[1:]+edges[:-1]); shell = np.pi*(edges[1:]**2 - edges[:-1]**2)
def pdist2d(a,b,Lx,Ly,same):
    dx = a[:,None,0]-b[None,:,0]; dy = a[:,None,1]-b[None,:,1]
    dx -= Lx*np.round(dx/Lx); dy -= Ly*np.round(dy/Ly)
    r = np.sqrt(dx*dx+dy*dy)
    return r[np.triu_indices(r.shape[0],1)] if same else r.ravel()
sel = {r: u.select_atoms(f"name {HEAD} and resname {r}") for r in present}
pairs = list(itertools.combinations_with_replacement(present,2))
hist = {p: np.zeros(NBINS) for p in pairs}; norm = {p: 0.0 for p in pairs}
for ts in u.trajectory[::STEP]:
    Lx, Ly = ts.dimensions[:2]; area = Lx*Ly
    xy = {r: sel[r].positions[:,:2] for r in present}
    for A,B in pairs:
        same = A==B
        hist[(A,B)] += np.histogram(pdist2d(xy[A],xy[B],Lx,Ly,same), bins=edges)[0]
        Na, Nb = len(xy[A]), len(xy[B])
        norm[(A,B)] += 0.5*Na*((Nb-1)/area) if same else Na*(Nb/area)
plt.figure(figsize=(7,4.5))
for A,B in pairs:
    if norm[(A,B)]==0: continue
    g = hist[(A,B)]/(norm[(A,B)]*shell)
    np.savetxt(f"{OUTDIR}/rdf_{A}-{B}.dat", np.column_stack([ctr/10,g]), header="r_nm g")
    plt.plot(ctr/10, g, label=f"{A}-{B}")
plt.axhline(1, color="k", lw=.6, ls="--")
plt.xlabel("r (nm)"); plt.ylabel("g(r)"); plt.legend(fontsize=8)
plt.grid(alpha=.3); plt.title("2D lateral RDF (P-P)"); plt.show()

## 11 &nbsp; Lipid lateral diffusion coefficient

Uses the **no-jump** trajectory and removes per-leaflet COM drift. Loads its own
Universe so it doesn't disturb the structural one above.


In [ ]:
ud = mda.Universe(TOPOLOGY, TRAJ_NOJUMP)
lip = ud.select_atoms("resname " + " ".join(present))
residues = lip.residues
resnames = np.array([r.resname for r in residues])
Pd = ud.select_atoms(f"name {HEAD} and resname " + " ".join(present))
ud.trajectory[0]; zc0 = Pd.positions[:,2].mean()
pz = {a.resid: a.position[2] for a in Pd}
leaf = np.array([0 if pz.get(r.resid, zc0) >= zc0 else 1 for r in residues])

times, coms = [], []
for ts in ud.trajectory[::STEP]:
    times.append(ts.time)
    com = np.array([r.atoms.center_of_mass()[:2] for r in residues])
    for lf in (0,1):
        m = leaf==lf
        if m.any(): com[m] -= com[m].mean(axis=0)
    coms.append(com)
times = np.array(times); coms = np.array(coms); T = len(times)
dt = np.mean(np.diff(times)) if T>1 else 1.0

FITMIN, FITMAX = 0.2, 0.6
plt.figure(figsize=(7,4.5))
print(f"{'lipid':<8}{'D (cm^2/s)':>14}{'D (1e-7 cm^2/s)':>18}")
for r in present:
    idx = np.where(resnames==r)[0]
    if idx.size==0: continue
    sub = coms[:,idx,:]; msd = np.zeros(T)
    for lag in range(1,T):
        msd[lag] = ((sub[lag:]-sub[:-lag])**2).sum(axis=2).mean()
    lags = np.arange(T)*dt
    np.savetxt(f"{OUTDIR}/msd_{r}.dat", np.column_stack([lags,msd]), header="lag_ps msd_A2")
    i0, i1 = max(1,int(FITMIN*T)), int(FITMAX*T)
    slope = np.polyfit(lags[i0:i1], msd[i0:i1], 1)[0]   # A^2/ps
    D = slope/4*1e-4                                     # -> cm^2/s
    print(f"{r:<8}{D:14.3e}{D*1e7:18.3f}")
    plt.plot(lags, msd, label=r)
plt.xlabel("lag (ps)"); plt.ylabel(r"MSD$_{xy}$ ($\AA^2$)")
plt.legend(); plt.grid(alpha=.3); plt.title("Lateral MSD"); plt.show()

## Preprocessing (run in a terminal before this notebook)

```bash
gmx trjconv -s md.tpr -f md.xtc        -o md_whole.xtc  -pbc whole
gmx trjconv -s md.tpr -f md_whole.xtc  -o md_center.xtc -pbc mol -center   # structural
gmx trjconv -s md.tpr -f md_whole.xtc  -o md_nojump.xtc -pbc nojump        # diffusion
```

**Per system:** set `LIPIDS = ["DOPC","DOPA"]` (or DOPG / DLPG / DLPA) and re-run.

**Caveats to report:** interaction energies are short-range Coul-SR + LJ-SR (PME
reciprocal part isn't group-decomposable — keep cutoffs identical across systems);
diffusion D is box-size dependent (Yeh–Hummer); validate Voronoi APL against the
box estimate printed in §7.
